In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"

In [44]:
# Get sequence of EGFR
with open(DATA_RAW / "UNIPROT_P00533.FASTA") as f:
    lines = f.read().splitlines()

header = lines[0]
seq = "".join(lines[1:])
len(seq)

# exclude peptide, only chain
seq = seq[24:] 

### Kinase domain
The full EGFR (human) sequence is given by Uniprot ID P00533.
However, the full 1210 residue protein is very large and only the kinase domain is relevant.
Here the kinase domain used in the crystal structure of the erlotinib complex [672–998] is used. Some key residues for checking alignment include:  
- Tyr845  
- Glu848  
- Lys836  

ref: Stamos et al. 2002 Structure of the Epidermal Growth Factor Receptor Kinase Domain
Alone and in Complex with a 4-Anilinoquinazoline Inhibitor

In [45]:
start, final = 672, 998

kinase_seq = seq[start-1:final]

In [47]:
kinase_seq

'GEAPNQALLRILKETEFKKIKVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLEDRRLVHRDLAARNVLVKTPQHVKITDFGLAKLLGAEEKEYHAEGGKVPIKWMALESILHRIYTHQSDVWSYGVTVWELMTFGSKPYDGIPASEISSILEKGERLPQPPICTIDVYMIMVKCWMIDADSRPKFRELIIEFSKMARDPQRYLVIQGDERMHLPSPTDSNFYRALMDEEDMDDVVDADEYLIPQQG'

In [49]:
import numpy as np
tyr = 845
glu = 848
lys = 836

# account for shift in indexing
print(f"Main chain residues: Tyr845 : {seq[tyr-1]}, Glu848 :{seq[glu-1]}, Lys836 : {seq[lys-1]}")
print(f"Kinase domain : Tyr845 : {kinase_seq[tyr-start]}, Glu848 :{kinase_seq[glu-start]}, Lys836 : {kinase_seq[lys-start]}")

Main chain residues: Tyr845 : Y, Glu848 :E, Lys836 : K
Kinase domain : Tyr845 : Y, Glu848 :E, Lys836 : K


In [50]:
import pandas as pd 

boltz_set = pd.read_csv(DATA_PROCESSED / "boltz_subset.csv")

In [61]:
# Make 1 yaml file per compound
import yaml

msa = ""

def write_yaml(smi, seq, name, msa=msa, dir=(PROJECT_ROOT / "boltz_inputs")):
    protein = {"id": "A", "sequence": seq}
    if msa is not None:
        protein["msa"] = str(msa)

    spec = {
        "version": 1,
        "sequences": [
            {"protein": protein},
            {"ligand": {"id": "B", "smiles": smi}},
        ],
        "properties": [{"affinity": {"binder": "B"}}],
    }

    path = PROJECT_ROOT / "boltz_inputs" / f"{name}.yaml"
    with open(path, "w") as f:
        yaml.safe_dump(spec, f, sort_keys=False, default_flow_style=False)
    return path

In [ ]:
# write 51 yamls
boltz_set.apply(lambda row : write_yaml(row["Clean Smiles"], kinase_seq, row["Molecule ChEMBL ID"]), axis=1)
